# LLM QLoRA Fine-Tuning on SG Hate Speech Dataset (Colab)

Fine-tune guard/SEA LLMs on the SG synthetic hate speech dataset using **Unsloth + QLoRA** on Google Colab.

**Runtime:** Colab Pro — T4 GPU (minimum) or A100 (recommended)  
**Task:** 3-class classification — `hate`, `offensive`, `normal`  
**Dataset:** SG synthetic, 319 train / 106 val / 108 test  
**Stack:** Unsloth + TRL SFTTrainer + bitsandbytes QLoRA (4-bit)

## Models
| Part | Model | Notes |
|------|-------|-------|
| 2 | `meta-llama/Llama-Guard-3-8B` | Guard model, Llama-3.1 base, fully Unsloth-supported |
| 3 | `aisingapore/sea-lion-7b-instruct` | SEA-focused, Llama-based |

## Structure
- **Part 0:** Colab setup (run once)
- **Part 1:** Data preparation (run once)
- **Part 2:** Llama-Guard-3-8B — train → eval → clear GPU
- **Part 3:** SEA-LION-7B — train → eval → clear GPU
- **Part 4:** Leaderboard comparison

> Parts 2 and 3 are independent — you can run either without the other.

---
# Part 0: Colab Setup

In [ ]:
# Cell 0-1: GPU Verification
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv

import torch
assert torch.cuda.is_available(), 'No GPU detected — change runtime to GPU'

gpu_name = torch.cuda.get_device_name(0)
gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f'GPU: {gpu_name}  |  VRAM: {gpu_mem:.1f} GB')
print(f'bfloat16 supported: {torch.cuda.is_bf16_supported()}')

In [ ]:
# Cell 0-2: Mount Google Drive (separate cell — do NOT combine with installs)
from google.colab import drive
drive.mount('/content/drive')
print('Drive mounted.')

In [ ]:
%%bash
# Cell 0-3: Install dependencies
pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
pip install -q trl bitsandbytes accelerate scikit-learn pandas
echo "Install complete."

In [ ]:
# Cell 0-4: Path setup — Drive or local fallback
from pathlib import Path

USE_DRIVE = True

try:
    DRIVE_ROOT = Path('/content/drive/MyDrive')
    assert DRIVE_ROOT.exists()
    DAP_HS_DIR = DRIVE_ROOT / 'dap-hs'
    DATA_DIR = DAP_HS_DIR / 'data' / 'singlish' / 'synthetic' / '20_terms' / 'split_60_20_20'
    OUTPUT_BASE = DAP_HS_DIR / 'lora_runs'
    print(f'Using Drive: {DAP_HS_DIR}')
except Exception:
    USE_DRIVE = False
    DAP_HS_DIR = Path('.')
    DATA_DIR = Path('data/singlish/synthetic/20_terms/split_60_20_20')
    OUTPUT_BASE = Path('runs/lora')
    print('Drive not available, using local paths')

OUTPUT_BASE.mkdir(parents=True, exist_ok=True)

# Verify data is accessible
for split in ['train', 'val', 'test']:
    p = DATA_DIR / f'{split}.jsonl'
    print(f'  {split}.jsonl: {"OK" if p.exists() else "MISSING — copy to Drive first"}')

In [ ]:
# Cell 0-5: HuggingFace token (needed for Llama-Guard gated model)
import os

try:
    from google.colab import userdata
    HF_TOKEN = userdata.get('HF_TOKEN')
    print('HF_TOKEN loaded from Colab secrets.')
except Exception:
    HF_TOKEN = os.getenv('HF_TOKEN', '')
    print('HF_TOKEN from env var (empty = unauthenticated).')

if not HF_TOKEN:
    print('WARNING: No HF token — Llama-Guard-3-8B is gated and will fail without it.')
    print('Add HF_TOKEN to Colab Secrets (key icon in left sidebar).')

---
# Part 1: Data Preparation

Convert SG JSONL to chat format for SFTTrainer. Run once — both models use the same formatted data.

In [ ]:
# Cell 1-1: Load SG splits
import json
from collections import Counter

def load_jsonl(path):
    with open(path) as f:
        return [json.loads(l) for l in f if l.strip()]

sg_train = load_jsonl(DATA_DIR / 'train.jsonl')
sg_val   = load_jsonl(DATA_DIR / 'val.jsonl')
sg_test  = load_jsonl(DATA_DIR / 'test.jsonl')

print(f'Train: {len(sg_train)}  Val: {len(sg_val)}  Test: {len(sg_test)}')
print(f'Train labels: {dict(Counter(d["label"] for d in sg_train))}')
print(f'\nSample: {sg_train[0]}')

In [ ]:
# Cell 1-2: Chat format conversion

SYSTEM_PROMPT = """You are a Singapore content moderation expert. Classify the following text into exactly one of:

- 'hate': targets a person or group based on race, religion, nationality, or identity. Includes racial slurs and xenophobic statements.
- 'offensive': rude or inappropriate language (e.g. Singlish expletives like 'nabeh', 'cb') not directed at an identity group.
- 'normal': neutral or benign content, including casual Singlish.

Reply with ONLY the label in lowercase: hate, offensive, or normal."""


def to_chat_format(row):
    """Convert a SG data row to the chat messages format expected by SFTTrainer."""
    return {
        'messages': [
            {'role': 'system',    'content': SYSTEM_PROMPT},
            {'role': 'user',      'content': f'Text: "{row["text"]}"'},
            {'role': 'assistant', 'content': row['label']},
        ]
    }


CHAT_TRAIN = [to_chat_format(r) for r in sg_train]
CHAT_VAL   = [to_chat_format(r) for r in sg_val]

# Save formatted JSONL to /content for fast local access during training
FORMATTED_DIR = Path('/content/sg_chat')
FORMATTED_DIR.mkdir(exist_ok=True)

for split_name, data in [('train', CHAT_TRAIN), ('val', CHAT_VAL)]:
    out_path = FORMATTED_DIR / f'{split_name}.jsonl'
    with open(out_path, 'w') as f:
        for row in data:
            f.write(json.dumps(row) + '\n')
    print(f'Saved {len(data)} rows to {out_path}')

print('\nSample chat format:')
print(json.dumps(CHAT_TRAIN[0], indent=2))

In [ ]:
# Cell 1-3: Load into HuggingFace Dataset
from datasets import load_dataset

chat_dataset = load_dataset(
    'json',
    data_files={'train': str(FORMATTED_DIR / 'train.jsonl'), 'validation': str(FORMATTED_DIR / 'val.jsonl')},
)
print(chat_dataset)
print('\nSample messages field:')
print(chat_dataset['train'][0]['messages'])

---
# Part 2: Llama-Guard-3-8B QLoRA

`meta-llama/Llama-Guard-3-8B` — Llama-3.1-based guard model.  
**Requires:** HF_TOKEN (gated model — request access at hf.co/meta-llama/Llama-Guard-3-8B first)  
**Estimated time:** ~45 min on T4, ~15 min on A100

In [ ]:
# Cell 2-1: Config
MODEL_A = 'meta-llama/Llama-Guard-3-8B'
ADAPTER_A = OUTPUT_BASE / 'llama-guard-3-8b-sg'
ADAPTER_A.mkdir(parents=True, exist_ok=True)

LORA_R = 16
LORA_ALPHA = 16
LORA_DROPOUT = 0.05
MAX_SEQ_LEN = 256
NUM_EPOCHS = 3
BATCH_SIZE = 2
GRAD_ACCUM = 4   # effective batch = 8
LR = 2e-4

print(f'Model:          {MODEL_A}')
print(f'Adapter output: {ADAPTER_A}')
print(f'Effective batch size: {BATCH_SIZE * GRAD_ACCUM}')

In [ ]:
# Cell 2-2: Load model with 4-bit QLoRA via Unsloth
from unsloth import FastLanguageModel

model_a, tokenizer_a = FastLanguageModel.from_pretrained(
    model_name=MODEL_A,
    max_seq_length=MAX_SEQ_LEN,
    dtype=None,       # Auto: bfloat16 on A100, float16 on T4
    load_in_4bit=True,
    token=HF_TOKEN,
)
print(f'Base model loaded. Trainable params before LoRA: {sum(p.numel() for p in model_a.parameters() if p.requires_grad):,}')

In [ ]:
# Cell 2-3: Add LoRA adapters
model_a = FastLanguageModel.get_peft_model(
    model_a,
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias='none',
    use_gradient_checkpointing='unsloth',
    random_state=42,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'up_proj', 'down_proj'],
)
trainable = sum(p.numel() for p in model_a.parameters() if p.requires_grad)
total = sum(p.numel() for p in model_a.parameters())
print(f'Trainable params: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)')

In [ ]:
# Cell 2-4: Train
from trl import SFTTrainer, SFTConfig

training_args_a = SFTConfig(
    output_dir=str(ADAPTER_A / 'checkpoints'),
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    per_device_eval_batch_size=BATCH_SIZE,
    warmup_ratio=0.1,
    learning_rate=LR,
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    logging_steps=10,
    eval_strategy='epoch',
    save_strategy='best',
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    optim='paged_adamw_8bit',
    weight_decay=0.01,
    lr_scheduler_type='cosine',
    seed=42,
    max_seq_length=MAX_SEQ_LEN,
    packing=False,
    dataset_text_field=None,  # Using messages format
)

trainer_a = SFTTrainer(
    model=model_a,
    tokenizer=tokenizer_a,
    train_dataset=chat_dataset['train'],
    eval_dataset=chat_dataset['validation'],
    args=training_args_a,
)

print(f'Training {MODEL_A.split("/")[-1]} for {NUM_EPOCHS} epochs...')
trainer_a.train()

# Save final adapter
model_a.save_pretrained(str(ADAPTER_A))
tokenizer_a.save_pretrained(str(ADAPTER_A))
print(f'Adapter saved to {ADAPTER_A}')

In [ ]:
# Cell 2-5: Evaluate on SG test set
from sklearn.metrics import classification_report, f1_score

LABEL_NAMES = ['hate', 'offensive', 'normal']

FastLanguageModel.for_inference(model_a)  # Enable fast inference mode

def classify_with_model(text, model, tokenizer, max_new_tokens=8):
    messages = [
        {'role': 'system',    'content': SYSTEM_PROMPT},
        {'role': 'user',      'content': f'Text: "{text}"'},
    ]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors='pt').to('cuda')
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False, pad_token_id=tokenizer.eos_token_id)
    generated = tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True).strip().lower()

    if generated.startswith('hate'):
        return 'hate'
    if generated.startswith('off'):
        return 'offensive'
    if generated.startswith('norm'):
        return 'normal'
    return 'invalid'


print(f'Evaluating {MODEL_A.split("/")[-1]} on SG test set ({len(sg_test)} rows)...')

from tqdm import tqdm as tqdm_bar

y_true_a, y_pred_a = [], []
for row in tqdm_bar(sg_test):
    pred = classify_with_model(row['text'], model_a, tokenizer_a)
    y_true_a.append(row['label'])
    y_pred_a.append(pred)

valid_a = [(t, p) for t, p in zip(y_true_a, y_pred_a) if p != 'invalid']
yt_a, yp_a = zip(*valid_a) if valid_a else ([], [])

print(f'\n{"="*60}')
print(f'  Llama-Guard-3-8B (fine-tuned) on SG Test Set')
print('='*60)
print(classification_report(yt_a, yp_a, labels=LABEL_NAMES, zero_division=0))
f1_a = f1_score(yt_a, yp_a, labels=LABEL_NAMES, average='macro', zero_division=0)
print(f'Macro F1: {f1_a:.4f}  |  Invalid predictions: {len(y_pred_a) - len(valid_a)}')

In [ ]:
# Cell 2-6: Save results + clear GPU
import csv
from datetime import datetime

RESULTS_FILE = str(DAP_HS_DIR / 'results' / 'model_scores_sg.csv')
os.makedirs(os.path.dirname(RESULTS_FILE), exist_ok=True)

report_a = classification_report(yt_a, yp_a, labels=LABEL_NAMES, output_dict=True, zero_division=0)
row_a = {
    'timestamp': datetime.now().isoformat(),
    'model_name': MODEL_A,
    'model_type': 'qlora (fine-tuned llm)',
    'task': 'sg_3class_lora',
    'epochs': NUM_EPOCHS,
    'lora_r': LORA_R,
    'test_samples': len(yt_a),
    'accuracy': round(report_a['accuracy'], 4),
    'macro_f1': round(f1_a, 4),
    'hate_f1': round(report_a.get('hate', {}).get('f1-score', 0), 4),
    'offensive_f1': round(report_a.get('offensive', {}).get('f1-score', 0), 4),
    'normal_f1': round(report_a.get('normal', {}).get('f1-score', 0), 4),
}

file_exists = os.path.exists(RESULTS_FILE)
with open(RESULTS_FILE, 'a', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=row_a.keys())
    if not file_exists:
        writer.writeheader()
    writer.writerow(row_a)
print(f'Results saved to {RESULTS_FILE}')

# Clear GPU before loading Model B
import gc
del model_a, tokenizer_a, trainer_a
gc.collect()
torch.cuda.empty_cache()
print(f'GPU cleared. Free VRAM: {torch.cuda.memory_reserved(0)/1e9:.1f} GB reserved')

---
# Part 3: SEA-LION-7B QLoRA

`aisingapore/sea-lion-7b-instruct` — SEA-focused LLM, Llama-based.  
**No token required** (public model)  
**Estimated time:** ~40 min on T4, ~12 min on A100

In [ ]:
# Cell 3-1: Config
MODEL_B = 'aisingapore/sea-lion-7b-instruct'
ADAPTER_B = OUTPUT_BASE / 'sea-lion-7b-sg'
ADAPTER_B.mkdir(parents=True, exist_ok=True)

# Same LoRA / training hyperparams as Model A
print(f'Model:          {MODEL_B}')
print(f'Adapter output: {ADAPTER_B}')

In [ ]:
# Cell 3-2: Load SEA-LION with Unsloth
# Note: If Unsloth raises an unsupported model error, fall back to raw PEFT below.
try:
    from unsloth import FastLanguageModel
    model_b, tokenizer_b = FastLanguageModel.from_pretrained(
        model_name=MODEL_B,
        max_seq_length=MAX_SEQ_LEN,
        dtype=None,
        load_in_4bit=True,
    )
    USE_UNSLOTH_B = True
    print('SEA-LION loaded via Unsloth.')
except Exception as e:
    print(f'Unsloth failed ({e}), falling back to raw PEFT...')
    from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
    from peft import get_peft_model, LoraConfig, TaskType

    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type='nf4',
    )
    tokenizer_b = AutoTokenizer.from_pretrained(MODEL_B)
    model_b = AutoModelForCausalLM.from_pretrained(MODEL_B, quantization_config=bnb_config, device_map='auto')
    lora_config = LoraConfig(
        r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT, bias='none',
        task_type=TaskType.CAUSAL_LM,
        target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'up_proj', 'down_proj'],
    )
    model_b = get_peft_model(model_b, lora_config)
    USE_UNSLOTH_B = False
    print('SEA-LION loaded via raw PEFT.')

trainable = sum(p.numel() for p in model_b.parameters() if p.requires_grad)
print(f'Trainable params: {trainable:,}')

In [ ]:
# Cell 3-3: Add LoRA (Unsloth path only — raw PEFT already added above)
if USE_UNSLOTH_B:
    model_b = FastLanguageModel.get_peft_model(
        model_b,
        r=LORA_R,
        lora_alpha=LORA_ALPHA,
        lora_dropout=LORA_DROPOUT,
        bias='none',
        use_gradient_checkpointing='unsloth',
        random_state=42,
        target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'up_proj', 'down_proj'],
    )
    print('LoRA added via Unsloth.')
else:
    print('LoRA already added via raw PEFT (skip this cell).')

In [ ]:
# Cell 3-4: Train SEA-LION
from trl import SFTTrainer, SFTConfig

training_args_b = SFTConfig(
    output_dir=str(ADAPTER_B / 'checkpoints'),
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    per_device_eval_batch_size=BATCH_SIZE,
    warmup_ratio=0.1,
    learning_rate=LR,
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    logging_steps=10,
    eval_strategy='epoch',
    save_strategy='best',
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    optim='paged_adamw_8bit',
    weight_decay=0.01,
    lr_scheduler_type='cosine',
    seed=42,
    max_seq_length=MAX_SEQ_LEN,
    packing=False,
    dataset_text_field=None,
)

trainer_b = SFTTrainer(
    model=model_b,
    tokenizer=tokenizer_b,
    train_dataset=chat_dataset['train'],
    eval_dataset=chat_dataset['validation'],
    args=training_args_b,
)

print(f'Training SEA-LION-7B for {NUM_EPOCHS} epochs...')
trainer_b.train()

model_b.save_pretrained(str(ADAPTER_B))
tokenizer_b.save_pretrained(str(ADAPTER_B))
print(f'Adapter saved to {ADAPTER_B}')

In [ ]:
# Cell 3-5: Evaluate SEA-LION on SG test set
if USE_UNSLOTH_B:
    FastLanguageModel.for_inference(model_b)

print(f'Evaluating SEA-LION-7B on SG test set ({len(sg_test)} rows)...')

y_true_b, y_pred_b = [], []
for row in tqdm_bar(sg_test):
    pred = classify_with_model(row['text'], model_b, tokenizer_b)
    y_true_b.append(row['label'])
    y_pred_b.append(pred)

valid_b = [(t, p) for t, p in zip(y_true_b, y_pred_b) if p != 'invalid']
yt_b, yp_b = zip(*valid_b) if valid_b else ([], [])

print(f'\n{"="*60}')
print(f'  SEA-LION-7B (fine-tuned) on SG Test Set')
print('='*60)
print(classification_report(yt_b, yp_b, labels=LABEL_NAMES, zero_division=0))
f1_b = f1_score(yt_b, yp_b, labels=LABEL_NAMES, average='macro', zero_division=0)
print(f'Macro F1: {f1_b:.4f}  |  Invalid: {len(y_pred_b) - len(valid_b)}')

In [ ]:
# Cell 3-6: Save results + clear GPU
report_b = classification_report(yt_b, yp_b, labels=LABEL_NAMES, output_dict=True, zero_division=0)
row_b = {
    'timestamp': datetime.now().isoformat(),
    'model_name': MODEL_B,
    'model_type': 'qlora (fine-tuned llm)',
    'task': 'sg_3class_lora',
    'epochs': NUM_EPOCHS,
    'lora_r': LORA_R,
    'test_samples': len(yt_b),
    'accuracy': round(report_b['accuracy'], 4),
    'macro_f1': round(f1_b, 4),
    'hate_f1': round(report_b.get('hate', {}).get('f1-score', 0), 4),
    'offensive_f1': round(report_b.get('offensive', {}).get('f1-score', 0), 4),
    'normal_f1': round(report_b.get('normal', {}).get('f1-score', 0), 4),
}

file_exists = os.path.exists(RESULTS_FILE)
with open(RESULTS_FILE, 'a', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=row_b.keys())
    if not file_exists:
        writer.writeheader()
    writer.writerow(row_b)
print(f'Results saved to {RESULTS_FILE}')

del model_b, tokenizer_b, trainer_b
gc.collect()
torch.cuda.empty_cache()
print('GPU cleared.')

---
# Part 4: Leaderboard

Compare all fine-tuned models against baselines from `llm_sg_baselines.ipynb`.

In [ ]:
import pandas as pd

# Load results from both notebooks (baselines + fine-tuned)
result_paths = [
    'results/model_scores_sg.csv',        # baseline notebook (local)
    str(DAP_HS_DIR / 'results' / 'model_scores_sg.csv'),  # Drive (fine-tuned)
]

dfs = []
for path in result_paths:
    if os.path.exists(path):
        dfs.append(pd.read_csv(path))

if not dfs:
    print('No results found. Run baselines notebook and at least one model above.')
else:
    df = pd.concat(dfs, ignore_index=True).drop_duplicates()
    df['short_model'] = df['model_name'].apply(lambda x: x.split('/')[-1])

    leaderboard = (
        df[['short_model', 'model_type', 'task', 'macro_f1', 'hate_f1', 'offensive_f1', 'normal_f1', 'accuracy']]
        .sort_values('macro_f1', ascending=False)
        .reset_index(drop=True)
    )
    leaderboard.index += 1

    print('SG Hate Speech — LLM Leaderboard (Macro F1 on 108 test rows)')
    print('='*90)
    print(leaderboard.to_string())